[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kauelopes/topicos-ml/blob/main/aulas/02-o-que-e-ml/notebooks/02_lab.ipynb)

# Aula 2 — Laboratório: crédito e perfis de consumo

**Parte A — Classificação (supervisionado):** prever inadimplência em pedidos de crédito.
**Parte B — Agrupamento (não supervisionado):** encontrar perfis de orçamento familiar.

> Os dados são **simulados** para fins didáticos: sabemos exatamente como foram gerados, o que nos permite checar o que o modelo aprende.
> Substitua cada `...` por código e responda às perguntas em **negrito**.

# Parte A — Crédito

## A0. Ficha de formulação
Preencha antes de olhar os dados:

| Item | Resposta |
|---|---|
| **Decisão** | |
| **Unidade** | |
| **Alvo y** | |
| **Atributos x** | |
| **Tarefa** | |
| **Métrica** | |

## A1. Os dados

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "legend.frameon": False})

import numpy as np
import pandas as pd

def gerar_credito(n=5000, seed=42):
    """Base SIMULADA de pedidos de crédito pessoal (didática)."""
    rng = np.random.default_rng(seed)
    renda = rng.lognormal(np.log(3.5), 0.55, n)               # R$ mil/mês
    idade = rng.integers(18, 75, n)
    divida_renda = np.clip(rng.beta(2, 5, n), 0, 0.95)         # comprometimento
    atrasos_ant = rng.poisson(0.4, n)                          # atrasos nos últimos 2 anos
    tempo_emprego = np.clip(rng.exponential(4, n), 0, 40)      # anos
    logit = (-2.4 - 0.25 * np.log(renda) - 0.02 * (idade - 40)
             + 4.0 * divida_renda + 0.7 * atrasos_ant - 0.06 * tempo_emprego)
    inadimplente = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)
    return pd.DataFrame(dict(renda=renda.round(2), idade=idade,
                             divida_renda=divida_renda.round(3),
                             atrasos_ant=atrasos_ant,
                             tempo_emprego=tempo_emprego.round(1),
                             inadimplente=inadimplente))

credito = gerar_credito()
credito.head()

In [ ]:
y = ...   # TODO: a coluna-alvo
X = ...   # TODO: os atributos (todas as outras colunas)
# TODO: qual a taxa de inadimplência na base?
...

## A2. Separar treino e teste

In [ ]:
from sklearn.model_selection import train_test_split
# TODO: separe 25% para teste. Use stratify=y. Por que isso é útil aqui?
X_tr, X_te, y_tr, y_te = ...
y_tr.mean(), y_te.mean()

## A3. Referência ingênua × regressão logística

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

ingenuo = ...   # TODO: DummyClassifier que prevê sempre a classe mais frequente
logit = ...     # TODO: LogisticRegression(max_iter=1000), treinada no treino

# TODO: compare a acurácia (método .score) dos dois no TESTE
...

**Pergunta:** a logística ganha da referência ingênua por muito ou por pouco? O que isso diz sobre a acurácia como métrica neste problema?

## A4. O modelo é útil mesmo assim?
O banco não precisa só de "sim/não": precisa **ordenar** os pedidos por risco.
Pegue os 10% de pedidos do teste com maior probabilidade prevista (`predict_proba`) e compare a taxa de inadimplência desse grupo com a taxa geral.

In [ ]:
p = ...        # TODO: probabilidade prevista de inadimplência (coluna 1 de predict_proba)
top10 = ...    # TODO: booleano, True para os 10% maiores p (dica: np.quantile)
# TODO: taxa de inadimplência geral × no grupo top10
...

## A4b. Visualize a separação
O modelo separa bem quem paga de quem não paga? Compare a distribuição de `p` nos dois grupos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
# TODO: dois histogramas de p no MESMO eixo: um para quem pagou (y_te == 0) e outro
#       para quem ficou inadimplente (y_te == 1). Use bins=30, alpha=0.6, density=True e label=...
...
# TODO: linhas verticais (ax.axvline) no limiar de 50% e no corte dos 10% mais arriscados
...
ax.set(xlabel="probabilidade prevista de inadimplência", ylabel="densidade")
ax.legend()
plt.show()

**Pergunta:** quase ninguém passa do limiar de 50%, e por isso o modelo quase nunca diz "inadimplente". Mesmo assim, ele **ordena** bem os pedidos. Por que a acurácia não enxerga isso? (Tema da Aula 3.)

## A5. Um modelo bom demais para ser verdade
Um colega sugere incluir a variável `cobrancas_enviadas`: o número de cartas de cobrança enviadas ao cliente **nos 12 meses após** o empréstimo.

In [ ]:
rng = np.random.default_rng(1)
credito["cobrancas_enviadas"] = rng.poisson(0.2 + 3.0 * credito["inadimplente"])

In [ ]:
# TODO: refaça A2–A3 incluindo a nova coluna e compare a acurácia
...

**Pergunta:** por que esse modelo seria inútil para o banco na hora de aprovar um pedido? Em qual item da ficha de formulação está o erro?

# Parte B — Perfis de orçamento (sem alvo)

In [ ]:
def gerar_orcamentos(n=600, seed=7):
    """Parcelas do orçamento de famílias SIMULADAS (somam 1)."""
    rng = np.random.default_rng(seed)
    perfis = {  # alimentação, moradia, transporte, lazer
        "A": [30, 40, 15, 5], "B": [18, 32, 25, 15], "C": [10, 28, 22, 30]}
    linhas = [rng.dirichlet(alpha) for alpha in perfis.values() for _ in range(n // 3)]
    return pd.DataFrame(linhas, columns=["alimentacao", "moradia", "transporte", "lazer"])

orc = gerar_orcamentos()
orc.head()

In [ ]:
from sklearn.cluster import KMeans
# TODO: ajuste um KMeans com 3 grupos às parcelas do orçamento,
#       guarde os rótulos em orc["grupo"] e mostre a média de cada variável por grupo
...

In [ ]:
vars_ = ["alimentacao", "moradia", "transporte", "lazer"]
# TODO: gráfico de barras com a média de cada parcela do orçamento por grupo.
# Dica: perfis = orc.groupby("grupo")[vars_].mean(); depois perfis.T.plot.barh(...)
#       e ax.xaxis.set_major_formatter(PercentFormatter(xmax=1)) para mostrar em %
...

**Pergunta:** dê um nome a cada grupo a partir das médias. Que tipo de família cada um poderia representar?

## B2. Quantos grupos?
Sem alvo, não há "resposta certa". Uma heurística: ver como a **inércia** (soma das distâncias ao centro do grupo) cai quando aumentamos `k`.

In [ ]:
import matplotlib.pyplot as plt
vars_ = ["alimentacao", "moradia", "transporte", "lazer"]
# TODO: para k de 1 a 7, ajuste um KMeans em orc[vars_] e guarde .inertia_
#       depois faça um gráfico de inércia × k
...

## Para refletir
1. Na Parte A, os coeficientes da regressão logística (`logit.coef_`) têm os sinais que você esperaria? Você os interpretaria como efeitos causais? Por quê?
2. Na Parte B, o que mudaria se você pedisse `k=6`? O algoritmo reclamaria?